# HR Statistical Analyis

## Load data

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/processed/HRDataset_v14_engineered.csv")

In [3]:
df.head()

,employee_name,employee_id,married_id,marital_status_id,gender_id,emp_status_id,dept_id,perf_score_id,from_diversity_job_fair_id,salary,...,age,years_since_review,age_at_hire,tenure_days,tenure_years,months_since_review,generation,absent_rate,lateness_flag,high_engagement_flag
0,"Adinolfi, Wilson K",10026,0,0,1,1,5,4,0,62506,...,36.476386,0.952772,27.986311,3101,8.490075,11.433265,Millennials,0.117785,True,False
1,"Ait Sidi, Karthikeyan",10084,1,1,1,5,3,3,0,104437,...,44.657084,3.849418,39.901437,444,1.215606,46.193018,Generation X,13.984797,True,False
2,"Akinkuolie, Sarah",10196,1,1,0,5,5,3,0,64955,...,31.279945,7.627652,22.789870,447,1.223819,91.531828,Millennials,2.451342,True,False
3,"Alagbe,Trina",10088,1,1,0,1,5,3,0,64991,...,31.258042,0.991102,19.277207,4376,11.980835,11.893224,Millennials,1.252000,True,False
4,"Anderson, Carol",10069,0,2,0,5,5,3,0,50825,...,30.310746,3.912389,21.837098,1884,5.158111,46.948665,Millennials,0.387739,True,False


## Analysis

### is engagement statistically different across departments?

In [16]:
# test if engagement is statistically different across departments
import scipy.stats as stats


# create a contingency table of department vs high_engagement_flag
contingency_table = pd.crosstab(df["department"], df["high_engagement_flag"])
print(contingency_table)

# perform chi-squared test
chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
print(f"Chi-squared: {chi2}, p-value: {p}")

high_engagement_flag  False  True 
department                        
Admin Offices             4      5
Executive Office          1      0
IT/IS                    10     40
Production              208      1
Sales                    31      0
Software Engineering      4      7
Chi-squared: 213.65023513659804, p-value: 3.4032870982652074e-44


In [19]:
# find the departments with the lowest (statistically significant) engagement using the chi-squared test results
import numpy as np


if p < 0.05:
    print("There is a statistically significant difference in engagement across departments.")
    # calculate the standardized residuals to find which departments are underperforming
    residuals = (contingency_table - expected) / np.sqrt(expected)
    print("Standardized Residuals:")
    print(residuals)
else:
    print("There is no statistically significant difference in engagement across departments.")

There is a statistically significant difference in engagement across departments.
Standardized Residuals:
high_engagement_flag     False      True 
department                               
Admin Offices        -1.268550   2.798848
Executive Office      0.187105  -0.412817
IT/IS                -4.887735  10.783992
Production            2.629004  -5.800469
Sales                 1.041758  -2.298469
Software Engineering -1.696688   3.743466


Looks like Production has quite a significant problem with engagement

In [35]:
# rerun the chi-squared test minus "Production" department to see if the other departments are significantly different from each other in terms of engagement
contingency_table_no_production = pd.crosstab(
    df[df["department"] != "Production"]["department"],
    df[df["department"] != "Production"]["high_engagement_flag"],
)
print(contingency_table_no_production)

chi2_no_production, p_no_production, dof_no_production, expected_no_production = stats.chi2_contingency(
    contingency_table_no_production
)
print(f"Chi-squared (no Production): {chi2_no_production}, p-value (no Production): {p_no_production}")

high_engagement_flag  False  True 
department                        
Admin Offices             4      5
Executive Office          1      0
IT/IS                    10     40
Sales                    31      0
Software Engineering      4      7
Chi-squared (no Production): 50.90965034965035, p-value (no Production): 2.3313892232396903e-10


In [36]:
# use the new chi-squared test results to find which departments are underperforming in terms of engagement
if p_no_production < 0.05:
    print("There is a statistically significant difference in engagement across departments (no Production).")
    residuals_no_production = (contingency_table_no_production - expected_no_production) / np.sqrt(
        expected_no_production
    )
    print("Standardized Residuals (no Production):")
    print(residuals_no_production)
else:
    print("There is no statistically significant difference in engagement across departments (no Production).")

There is a statistically significant difference in engagement across departments (no Production).
Standardized Residuals (no Production):
high_engagement_flag     False     True 
department                              
Admin Offices        -0.196039  0.192232
Executive Office      0.728146 -0.714006
IT/IS                -2.930837  2.873922
Sales                 4.054143 -3.975415
Software Engineering -0.599524  0.587882


We now see Production has the most significant issue, followed by Sales

IT/IS stands out as the most engaged department

### is attrition associated with salary?

In [21]:
df.columns

Index(['employee_name', 'employee_id', 'married_id', 'marital_status_id',
       'gender_id', 'emp_status_id', 'dept_id', 'perf_score_id',
       'from_diversity_job_fair_id', 'salary', 'termd', 'position_id',
       'position', 'state', 'zip', 'dob', 'sex', 'marital_desc',
       'citizen_desc', 'hispanic_latino', 'race_desc', 'date_of_hire',
       'date_of_termination', 'term_reason', 'employment_status', 'department',
       'manager_name', 'manager_id', 'recruitment_source', 'performance_score',
       'engagement_survey', 'emp_satisfaction', 'special_projects_count',
       'last_performance_review_date', 'days_late_last_30', 'absences',
       'salary_band', 'engagement_quartile', 'quality_of_hire', 'tenure',
       'salary_relative_to_dept', 'promoted', 'age', 'years_since_review',
       'age_at_hire', 'tenure_days', 'tenure_years', 'months_since_review',
       'generation', 'absent_rate', 'lateness_flag', 'high_engagement_flag'],
      dtype='object')

In [27]:
# attrition (termd) association with salary (salary_band)
contingency_table = pd.crosstab(df["termd"], df["salary_band"])
print(contingency_table)

chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
print(f"Chi-squared: {chi2}, p-value: {p}")

salary_band  0-50k  100k-150k  150k-200k  200k+  50k-100k
termd                                                    
0               17          9          5      2       174
1               14          9          0      0        81
Chi-squared: 7.969586486192996, p-value: 0.0926986549382612


Looks like attrition and salary bands don't have a statistical difference

### are diversity hiring sources statistically different?

In [32]:
# is there are statistically significant difference in attrition and those hired through "Diversity Job Fair"
# create a column for whether the recruitment source is "Diversity Job Fair"
df["recruitment_diversity_job_fair"] = df["recruitment_source"] == "Diversity Job Fair"

contingency_table = pd.crosstab(df["termd"], df["recruitment_diversity_job_fair"])
print(contingency_table)

chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
print(f"Chi-squared: {chi2}, p-value: {p}")

recruitment_diversity_job_fair  False  True 
termd                                       
0                                 194     13
1                                  88     16
Chi-squared: 5.752059513921434, p-value: 0.016469337855285995


In [33]:
# use chi-squared test results to determine if there is a statistically significant difference in attrition based on recruitment source
if p < 0.05:
    print("There is a statistically significant difference in attrition based on recruitment source.")
    # calculate the standardized residuals to find which recruitment sources are underperforming
    residuals = (contingency_table - expected) / np.sqrt(expected)
    print("Standardized Residuals:")
    print(residuals)
else:
    print("There is no statistically significant difference in attrition based on recruitment source.")

There is a statistically significant difference in attrition based on recruitment source.
Standardized Residuals:
recruitment_diversity_job_fair     False     True 
termd                                             
0                               0.460009 -1.434471
1                              -0.648985  2.023766


It appears that those recruited by the diversity job fairs could have an issue with attrition, but the sample size and signifigance are to be noted